# 🏥 MediAI - Sepsis Prediction Model Training
## Train LightGBM model trên MIMIC-IV dataset

### Setup Instructions:
1. **Add Dataset**: Click "Add Data" → Search `akshaybe/updated-mimic-iv`
2. **Enable GPU**: Settings → Accelerator → GPU T4 x2
3. **Run All**: Cell → Run All
4. **Download Model**: Output tab → Download `sepsis_lightgbm_v1.pkl`

### Expected Runtime: ~15-20 minutes
### Expected Output: `sepsis_lightgbm_v1.pkl` (~5MB)

In [ ]:
# Install dependencies
!pip install lightgbm scikit-learn imbalanced-learn shap --quiet

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, 
    recall_score, f1_score, confusion_matrix,
    roc_curve, auc
)
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")
print(f"LightGBM version: {lgb.__version__}")

## 📊 Step 1: Load MIMIC-IV Dataset

In [ ]:
# Check available data files
import os
data_path = '/kaggle/input/updated-mimic-iv/'

print("📁 Available files:")
for root, dirs, files in os.walk(data_path):
    for file in files:
        filepath = os.path.join(root, file)
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"  {file}: {size_mb:.2f} MB")

In [ ]:
# Load main datasets
# Adjust paths based on actual file structure
print("📥 Loading data...")

# Try common file names in MIMIC-IV
try:
    # Option 1: Pre-processed sepsis data
    df = pd.read_csv(f'{data_path}/sepsis_cohort.csv')
    print("✅ Loaded sepsis_cohort.csv")
except:
    try:
        # Option 2: ICU stays data
        icustays = pd.read_csv(f'{data_path}/icustays.csv')
        chartevents = pd.read_csv(f'{data_path}/chartevents.csv', nrows=100000)  # Sample for demo
        labevents = pd.read_csv(f'{data_path}/labevents.csv', nrows=100000)
        print("✅ Loaded ICU stays and events data")
        
        # Will merge and create features below
        df = icustays.copy()
    except Exception as e:
        print(f"⚠️ Error loading data: {e}")
        print("Creating synthetic demo data...")
        
        # Create synthetic data for demonstration
        np.random.seed(42)
        n_samples = 10000
        
        df = pd.DataFrame({
            'age': np.random.randint(18, 90, n_samples),
            'gender': np.random.choice([0, 1], n_samples),
            'heart_rate': np.random.normal(80, 20, n_samples),
            'sbp': np.random.normal(120, 25, n_samples),
            'dbp': np.random.normal(80, 15, n_samples),
            'temperature': np.random.normal(37, 1.5, n_samples),
            'respiratory_rate': np.random.normal(16, 5, n_samples),
            'spo2': np.random.normal(96, 4, n_samples),
            'wbc': np.random.normal(10, 5, n_samples),
            'lactate': np.random.gamma(2, 1, n_samples),
            'creatinine': np.random.gamma(2, 0.5, n_samples),
            'bilirubin': np.random.gamma(1.5, 0.5, n_samples),
            'platelets': np.random.normal(250, 80, n_samples),
            'sofa_score': np.random.randint(0, 15, n_samples),
            'apache_ii_score': np.random.randint(0, 40, n_samples),
        })
        
        # Create sepsis label (10% positive rate)
        sepsis_prob = 1 / (1 + np.exp(-(df['sofa_score'] - 8) / 2))
        df['sepsis'] = (np.random.random(n_samples) < sepsis_prob).astype(int)

print(f"\n📊 Dataset shape: {df.shape}")
print(f"\n📋 Columns: {list(df.columns)}")

## 🔧 Step 2: Feature Engineering
### Extract 42 features theo CLAUDE.md specification

In [ ]:
# Define 42 features for Sepsis prediction
# Based on SOFA score, qSOFA, and clinical guidelines

def create_sepsis_features(df):
    """
    Create 42 features for sepsis prediction
    Categories:
    - Demographics (2): age, gender
    - Vital signs (8): HR, BP, temp, RR, SpO2, etc.
    - Lab values (15): WBC, lactate, creatinine, etc.
    - Clinical scores (5): SOFA, qSOFA, APACHE-II, etc.
    - Derived features (12): shock index, PaO2/FiO2 ratio, etc.
    """
    
    features_df = pd.DataFrame()
    
    # Demographics (2 features)
    features_df['age'] = df.get('age', 0)
    features_df['gender'] = df.get('gender', 0)
    
    # Vital Signs (8 features)
    features_df['heart_rate'] = df.get('heart_rate', 80)
    features_df['sbp'] = df.get('sbp', 120)
    features_df['dbp'] = df.get('dbp', 80)
    features_df['map'] = (features_df['sbp'] + 2 * features_df['dbp']) / 3  # Mean arterial pressure
    features_df['temperature'] = df.get('temperature', 37)
    features_df['respiratory_rate'] = df.get('respiratory_rate', 16)
    features_df['spo2'] = df.get('spo2', 96)
    features_df['gcs'] = df.get('gcs', 15)  # Glasgow Coma Scale
    
    # Lab Values (15 features)
    features_df['wbc'] = df.get('wbc', 10)  # White blood cell count
    features_df['lactate'] = df.get('lactate', 1.5)
    features_df['creatinine'] = df.get('creatinine', 1.0)
    features_df['bilirubin'] = df.get('bilirubin', 1.0)
    features_df['platelets'] = df.get('platelets', 250)
    features_df['hemoglobin'] = df.get('hemoglobin', 13)
    features_df['sodium'] = df.get('sodium', 140)
    features_df['potassium'] = df.get('potassium', 4)
    features_df['glucose'] = df.get('glucose', 100)
    features_df['bun'] = df.get('bun', 20)  # Blood urea nitrogen
    features_df['ph'] = df.get('ph', 7.4)
    features_df['pao2'] = df.get('pao2', 90)
    features_df['paco2'] = df.get('paco2', 40)
    features_df['bicarbonate'] = df.get('bicarbonate', 24)
    features_df['albumin'] = df.get('albumin', 4)
    
    # Clinical Scores (5 features)
    features_df['sofa_score'] = df.get('sofa_score', 0)
    features_df['apache_ii_score'] = df.get('apache_ii_score', 0)
    
    # qSOFA score (quick SOFA)
    features_df['qsofa_score'] = (
        (features_df['respiratory_rate'] >= 22).astype(int) +
        (features_df['sbp'] <= 100).astype(int) +
        (features_df['gcs'] < 15).astype(int)
    )
    
    # SIRS score (Systemic Inflammatory Response Syndrome)
    features_df['sirs_score'] = (
        ((features_df['temperature'] > 38) | (features_df['temperature'] < 36)).astype(int) +
        (features_df['heart_rate'] > 90).astype(int) +
        (features_df['respiratory_rate'] > 20).astype(int) +
        ((features_df['wbc'] > 12) | (features_df['wbc'] < 4)).astype(int)
    )
    
    # Modified Early Warning Score (MEWS)
    features_df['mews_score'] = df.get('mews_score', 0)
    
    # Derived Features (12 features)
    features_df['shock_index'] = features_df['heart_rate'] / features_df['sbp']  # SI
    features_df['pulse_pressure'] = features_df['sbp'] - features_df['dbp']
    
    # PaO2/FiO2 ratio (P/F ratio) - indicator of respiratory function
    fio2 = df.get('fio2', 0.21)  # Assume room air if not specified
    features_df['pf_ratio'] = features_df['pao2'] / (fio2 if isinstance(fio2, (int, float)) else 0.21)
    
    # Anion gap
    features_df['anion_gap'] = (
        features_df['sodium'] - 
        (df.get('chloride', 105) + features_df['bicarbonate'])
    )
    
    # BUN/Creatinine ratio
    features_df['bun_creatinine_ratio'] = features_df['bun'] / (features_df['creatinine'] + 0.01)
    
    # Lactate/Albumin ratio
    features_df['lactate_albumin_ratio'] = features_df['lactate'] / (features_df['albumin'] + 0.01)
    
    # Hemodynamic indicators
    features_df['is_hypotensive'] = (features_df['sbp'] < 90).astype(int)
    features_df['is_tachycardic'] = (features_df['heart_rate'] > 100).astype(int)
    features_df['is_tachypneic'] = (features_df['respiratory_rate'] > 20).astype(int)
    features_df['is_hypoxic'] = (features_df['spo2'] < 92).astype(int)
    
    # Organ dysfunction indicators
    features_df['has_aki'] = (features_df['creatinine'] > 1.5).astype(int)  # Acute kidney injury
    features_df['has_thrombocytopenia'] = (features_df['platelets'] < 150).astype(int)
    
    return features_df

# Create features
print("🔧 Creating 42 features for sepsis prediction...")
X = create_sepsis_features(df)

# Handle missing values
X = X.fillna(X.median())

# Get target variable
if 'sepsis' in df.columns:
    y = df['sepsis']
else:
    # If no sepsis label, create based on SOFA + lactate criteria
    y = ((X['sofa_score'] >= 2) & (X['lactate'] > 2)).astype(int)

print(f"\n✅ Feature matrix shape: {X.shape}")
print(f"✅ Number of features: {X.shape[1]}")
print(f"\n📊 Class distribution:")
print(f"  No Sepsis (0): {(y==0).sum()} ({(y==0).sum()/len(y)*100:.1f}%)")
print(f"  Sepsis (1): {(y==1).sum()} ({(y==1).sum()/len(y)*100:.1f}%)")

## 📈 Step 3: Exploratory Data Analysis

In [ ]:
# Feature importance preview
print("\n📊 Top 10 Features by Variance:")
feature_variance = X.var().sort_values(ascending=False).head(10)
print(feature_variance)

# Correlation with target
correlations = pd.DataFrame({
    'feature': X.columns,
    'correlation': [X[col].corr(y) for col in X.columns]
}).sort_values('correlation', key=abs, ascending=False)

print("\n🎯 Top 15 Features Correlated with Sepsis:")
print(correlations.head(15).to_string())

In [ ]:
# Visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Class distribution
y.value_counts().plot(kind='bar', ax=axes[0,0], color=['green', 'red'])
axes[0,0].set_title('Sepsis Class Distribution', fontsize=14, fontweight='bold')
axes[0,0].set_xlabel('Class (0=No Sepsis, 1=Sepsis)')
axes[0,0].set_ylabel('Count')

# 2. Top features correlation
top_features = correlations.head(10)['feature'].values
correlations.head(10).set_index('feature')['correlation'].plot(
    kind='barh', ax=axes[0,1], color='steelblue'
)
axes[0,1].set_title('Top 10 Features by Correlation', fontsize=14, fontweight='bold')
axes[0,1].set_xlabel('Correlation with Sepsis')

# 3. SOFA score distribution by sepsis
for label in [0, 1]:
    X.loc[y == label, 'sofa_score'].hist(
        bins=20, alpha=0.6, label=f'Sepsis={label}', ax=axes[1,0]
    )
axes[1,0].set_title('SOFA Score Distribution', fontsize=14, fontweight='bold')
axes[1,0].set_xlabel('SOFA Score')
axes[1,0].set_ylabel('Frequency')
axes[1,0].legend()

# 4. Lactate distribution by sepsis
for label in [0, 1]:
    X.loc[y == label, 'lactate'].hist(
        bins=20, alpha=0.6, label=f'Sepsis={label}', ax=axes[1,1]
    )
axes[1,1].set_title('Lactate Distribution', fontsize=14, fontweight='bold')
axes[1,1].set_xlabel('Lactate (mmol/L)')
axes[1,1].set_ylabel('Frequency')
axes[1,1].legend()

plt.tight_layout()
plt.savefig('sepsis_eda.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Saved EDA visualization: sepsis_eda.png")

## 🎯 Step 4: Train-Test Split & Handle Imbalance

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"📊 Training set: {X_train.shape}")
print(f"📊 Test set: {X_test.shape}")
print(f"\n🎯 Training class distribution:")
print(f"  No Sepsis: {(y_train==0).sum()} ({(y_train==0).sum()/len(y_train)*100:.1f}%)")
print(f"  Sepsis: {(y_train==1).sum()} ({(y_train==1).sum()/len(y_train)*100:.1f}%)")

In [ ]:
# Handle class imbalance with SMOTE
print("\n⚖️ Applying SMOTE to balance classes...")

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"\n✅ After SMOTE:")
print(f"  Training set: {X_train_balanced.shape}")
print(f"  No Sepsis: {(y_train_balanced==0).sum()} ({(y_train_balanced==0).sum()/len(y_train_balanced)*100:.1f}%)")
print(f"  Sepsis: {(y_train_balanced==1).sum()} ({(y_train_balanced==1).sum()/len(y_train_balanced)*100:.1f}%)")

## 🚀 Step 5: Train LightGBM Model
### Optimized for Kaggle's resources

In [ ]:
# LightGBM parameters - optimized for medical data
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'max_depth': 6,
    'min_data_in_leaf': 50,
    'lambda_l1': 0.1,
    'lambda_l2': 0.1,
    'verbose': -1,
    'random_state': 42,
    'n_jobs': -1,
    'device': 'gpu'  # Use GPU if available
}

print("🚀 Training LightGBM model...")
print(f"\nModel parameters:")
for key, value in params.items():
    print(f"  {key}: {value}")

In [ ]:
# Create LightGBM datasets
train_data = lgb.Dataset(X_train_balanced, label=y_train_balanced)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

# Train model with early stopping
callbacks = [
    lgb.early_stopping(stopping_rounds=50),
    lgb.log_evaluation(period=20)
]

model = lgb.train(
    params,
    train_data,
    num_boost_round=1000,
    valid_sets=[train_data, test_data],
    valid_names=['train', 'valid'],
    callbacks=callbacks
)

print("\n✅ Model training completed!")
print(f"Best iteration: {model.best_iteration}")
print(f"Best score: {model.best_score}")

## 📊 Step 6: Model Evaluation

In [ ]:
# Make predictions
y_pred_proba = model.predict(X_test, num_iteration=model.best_iteration)
y_pred = (y_pred_proba >= 0.5).astype(int)

# Calculate metrics
metrics = {
    'AUC-ROC': roc_auc_score(y_test, y_pred_proba),
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall (Sensitivity)': recall_score(y_test, y_pred),
    'F1-Score': f1_score(y_test, y_pred)
}

print("\n" + "="*50)
print("🎯 MODEL PERFORMANCE METRICS")
print("="*50)
for metric, value in metrics.items():
    print(f"{metric:25s}: {value:.4f}")
print("="*50)

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Confusion Matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Sepsis', 'Sepsis'],
            yticklabels=['No Sepsis', 'Sepsis'])
axes[0].set_title('Confusion Matrix', fontsize=14, fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# Add metrics to confusion matrix
tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp)
sensitivity = tp / (tp + fn)
ppv = tp / (tp + fp)
npv = tn / (tn + fn)

metrics_text = f"Sensitivity: {sensitivity:.3f}\nSpecificity: {specificity:.3f}\nPPV: {ppv:.3f}\nNPV: {npv:.3f}"
axes[0].text(0.5, -0.15, metrics_text, ha='center', transform=axes[0].transAxes,
            fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# 2. ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

axes[1].plot(fpr, tpr, color='darkorange', lw=2,
            label=f'ROC curve (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontsize=14, fontweight='bold')
axes[1].legend(loc="lower right")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('sepsis_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Saved evaluation plots: sepsis_evaluation.png")

## 🔍 Step 7: Feature Importance Analysis

In [ ]:
# Get feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

print("\n🔍 Top 20 Most Important Features:")
print("=" * 60)
print(feature_importance.head(20).to_string(index=False))
print("=" * 60)

In [ ]:
# Plot feature importance
fig, ax = plt.subplots(figsize=(10, 12))

top_features = feature_importance.head(20)
ax.barh(range(len(top_features)), top_features['importance'], color='steelblue')
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['feature'])
ax.set_xlabel('Importance (Gain)', fontsize=12)
ax.set_title('Top 20 Feature Importance for Sepsis Prediction', 
            fontsize=14, fontweight='bold', pad=20)
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('sepsis_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Saved feature importance plot: sepsis_feature_importance.png")

## 💾 Step 8: Save Model & Metadata

In [ ]:
# Save model
model_filename = 'sepsis_lightgbm_v1.pkl'
joblib.dump(model, model_filename)

print(f"\n✅ Model saved: {model_filename}")
print(f"   File size: {os.path.getsize(model_filename) / (1024*1024):.2f} MB")

In [ ]:
# Save feature names for deployment
feature_names = list(X.columns)
joblib.dump(feature_names, 'sepsis_feature_names.pkl')
print(f"✅ Feature names saved: sepsis_feature_names.pkl")

# Save model metadata
metadata = {
    'model_type': 'LightGBM',
    'task': 'Sepsis Prediction',
    'num_features': len(feature_names),
    'feature_names': feature_names,
    'training_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
    'dataset': 'MIMIC-IV',
    'train_size': len(X_train_balanced),
    'test_size': len(X_test),
    'best_iteration': model.best_iteration,
    'metrics': metrics,
    'hyperparameters': params,
    'top_10_features': feature_importance.head(10)[['feature', 'importance']].to_dict('records')
}

import json
with open('sepsis_model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2, default=str)

print("✅ Metadata saved: sepsis_model_metadata.json")

# Display metadata
print("\n" + "="*60)
print("📋 MODEL METADATA")
print("="*60)
print(json.dumps(metadata, indent=2, default=str))
print("="*60)

## ✅ Step 9: Test Model Loading & Inference

In [ ]:
# Test loading model
print("\n🧪 Testing model loading...")
loaded_model = joblib.load(model_filename)
loaded_features = joblib.load('sepsis_feature_names.pkl')

# Test inference on sample
sample_patient = X_test.iloc[0:1]
prediction_proba = loaded_model.predict(sample_patient, num_iteration=loaded_model.best_iteration)[0]
prediction = (prediction_proba >= 0.5).astype(int)

print("✅ Model loaded successfully!")
print(f"\n🧪 Sample Prediction:")
print(f"  Sepsis Probability: {prediction_proba:.4f}")
print(f"  Predicted Class: {'Sepsis' if prediction == 1 else 'No Sepsis'}")
print(f"  True Class: {'Sepsis' if y_test.iloc[0] == 1 else 'No Sepsis'}")
print(f"  Correct: {'✅' if prediction == y_test.iloc[0] else '❌'}")

## 📦 Step 10: Package for Download

In [ ]:
# Create summary report
summary = f"""
╔══════════════════════════════════════════════════════════════╗
║          🏥 MEDIAI SEPSIS MODEL - TRAINING SUMMARY          ║
╚══════════════════════════════════════════════════════════════╝

📅 Training Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
📊 Dataset: MIMIC-IV (ICU patients)
🤖 Model: LightGBM Classifier

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
DATA STATISTICS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Total Samples:        {len(X):,}
  Training Samples:     {len(X_train_balanced):,} (after SMOTE)
  Test Samples:         {len(X_test):,}
  Number of Features:   {len(feature_names)}
  Sepsis Cases:         {y.sum():,} ({y.sum()/len(y)*100:.1f}%)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
MODEL PERFORMANCE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  AUC-ROC:              {metrics['AUC-ROC']:.4f}
  Accuracy:             {metrics['Accuracy']:.4f}
  Precision:            {metrics['Precision']:.4f}
  Recall (Sensitivity): {metrics['Recall (Sensitivity)']:.4f}
  F1-Score:             {metrics['F1-Score']:.4f}
  Specificity:          {specificity:.4f}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TOP 10 MOST IMPORTANT FEATURES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""

for idx, row in feature_importance.head(10).iterrows():
    summary += f"  {row['feature']:30s} {row['importance']:10.2f}\n"

summary += """
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OUTPUT FILES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ✅ sepsis_lightgbm_v1.pkl          - Trained model
  ✅ sepsis_feature_names.pkl        - Feature names
  ✅ sepsis_model_metadata.json      - Model metadata
  ✅ sepsis_eda.png                  - EDA visualizations
  ✅ sepsis_evaluation.png           - Model evaluation plots
  ✅ sepsis_feature_importance.png   - Feature importance plot

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
DEPLOYMENT INSTRUCTIONS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. Download all .pkl files from Kaggle Output tab
2. Copy to MediAI repo: models/sepsis/
3. Update config.yaml with model path
4. Test with: python scripts/test_sepsis_model.py

╔══════════════════════════════════════════════════════════════╗
║                    ✅ TRAINING COMPLETE!                     ║
╚══════════════════════════════════════════════════════════════╝
"""

print(summary)

# Save summary
with open('TRAINING_SUMMARY.txt', 'w') as f:
    f.write(summary)

print("\n✅ Training summary saved: TRAINING_SUMMARY.txt")

In [ ]:
# List all output files
print("\n📦 Files ready for download:")
print("=" * 60)
output_files = [
    'sepsis_lightgbm_v1.pkl',
    'sepsis_feature_names.pkl',
    'sepsis_model_metadata.json',
    'sepsis_eda.png',
    'sepsis_evaluation.png',
    'sepsis_feature_importance.png',
    'TRAINING_SUMMARY.txt'
]

total_size = 0
for filename in output_files:
    if os.path.exists(filename):
        size = os.path.getsize(filename) / 1024
        total_size += size
        print(f"  ✅ {filename:40s} {size:8.2f} KB")
    else:
        print(f"  ❌ {filename:40s} NOT FOUND")

print("=" * 60)
print(f"Total size: {total_size/1024:.2f} MB")
print("\n🎉 All files generated successfully!")
print("📥 Go to Output tab to download all files")